In [3]:
from merge_tables.db.connection import connect_to_postgres_via_duckdb

from pathlib import Path


MERGE_TABLES_DIR = Path('.').parent
DATA_DIR = MERGE_TABLES_DIR / "data"
OUTPUT_DIR = MERGE_TABLES_DIR / "output"

OUTPUT_DIR.mkdir(exist_ok=True)

In [4]:
duck = connect_to_postgres_via_duckdb()

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'


In [25]:
duck.sql(
    r"""
    create or replace table medisoft_besch as 
        select rec_id, lower(vorname) as vorname, lower(familienname) as familienname, telefon_mobil, regexp_extract(identifikation, '(\d{2}\.\d{2}\.\d{4})') as geburtsdatum from pg.medisoft.table_beschaeftigte
    """
)

In [44]:
duck.sql(
    """
    select * from medisoft_besch
    """
).to_csv('medisoft_besch.csv')

In [40]:
duck.sql(
    """
    select  count(*)
    from read_csv('/Users/adrienblanquer/code/bas-utils/patients_2026-03-10_15-51-44.csv')
    join medisoft_besch 
        on strip_accents(lower(last_name)) = strip_accents(familienname) and strip_accents(lower(first_name)) = strip_accents(vorname)
    """
)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        11675 │
└──────────────┘

In [ ]:
duck.sql(
    r"""
    with patients as (
        select
            row_number() over () as patient_id,
            regexp_extract(birthdate, '(\d{2}\.\d{2}\.\d{4})') as birthdate_clean,
            *
        from read_csv('/Users/adrienblanquer/code/bas-utils/patients_2026-03-10_15-51-44.csv')
    ),
    matches as (
        select
            p.patient_id,
            p.city,
            p.last_name,
            p.first_name,
            p.phone,
            p.mobile,
            p.email,
            p.address,
            p.birthdate,
            m.rec_id,
            m.telefon_mobil,
            m.geburtsdatum as geburtsdatum_medisoft,
            case
                when p.birthdate_clean != '' and p.birthdate_clean = m.geburtsdatum then 1
                else 0
            end as birthdate_match
        from patients p
        join medisoft_besch m
            on strip_accents(lower(p.last_name))  = strip_accents(m.familienname)
           and strip_accents(lower(p.first_name)) = strip_accents(m.vorname)
    )
    select * exclude (birthdate_match)
    from matches
    qualify row_number() over (partition by patient_id order by birthdate_match desc) = 1
    order by patient_id
    """
)

┌────────────┬─────────┬────────────┬────────────┬──────────────┬──────────────┬────────────────────────────┬─────────┬───────────────────────┬──────────────────────────────────────┬───────────────┬───────────────────────┐
│ patient_id │  city   │ last_name  │ first_name │    phone     │    mobile    │           email            │ address │       birthdate       │                rec_id                │ telefon_mobil │ geburtsdatum_medisoft │
│   int64    │ varchar │  varchar   │  varchar   │   varchar    │   varchar    │          varchar           │ varchar │        varchar        │               varchar                │    varchar    │        varchar        │
├────────────┼─────────┼────────────┼────────────┼──────────────┼──────────────┼────────────────────────────┼─────────┼───────────────────────┼──────────────────────────────────────┼───────────────┼───────────────────────┤
│          2 │ Rostock │ Schwerdt   │ Michael    │ NULL         │ NULL         │ NULL                       